In [1]:
pip install pandas

  Using cached pandas-3.0.5-cp313-cp313-win_amd64.whl.metadata (19 kB)
Using cached pandas-3.0.5-cp313-cp313-win_amd64.whl (9.8 MB)
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import pandas as pd
import numpy as np
from pathlib import Path

# Project root
ROOT = Path.cwd().parent

TRANSACTION_PATH = ROOT / "data" / "raw" / "train_transaction.csv"
IDENTITY_PATH = ROOT / "data" / "raw" / "train_identity.csv"

print("Transaction file:", TRANSACTION_PATH)
print("Identity file:", IDENTITY_PATH)

print("Transaction exists:", TRANSACTION_PATH.exists())
print("Identity exists:", IDENTITY_PATH.exists())

Transaction file: c:\Users\Gaurav Sehgal\OneDrive\Desktop\financial_fraud_detection_agent\data\raw\train_transaction.csv
Identity file: c:\Users\Gaurav Sehgal\OneDrive\Desktop\financial_fraud_detection_agent\data\raw\train_identity.csv
Transaction exists: True
Identity exists: True


In [3]:
transactions = pd.read_csv(TRANSACTION_PATH)
identity = pd.read_csv(IDENTITY_PATH)

print("Transaction shape:", transactions.shape)
print("Identity shape:", identity.shape)

Transaction shape: (590540, 394)
Identity shape: (144233, 41)


In [4]:
transactions["isFraud"].value_counts()

isFraud
0    569877
1     20663
Name: count, dtype: int64

In [5]:
fraud_rate = transactions["isFraud"].mean() * 100

print(f"Fraud rate: {fraud_rate:.2f}%")

Fraud rate: 3.50%


In [6]:
missing = (
    transactions.isna()
    .mean()
    .mul(100)
    .sort_values(ascending=False)
)

missing.head(30)

dist2    93.628374
D7       93.409930
D13      89.509263
D14      89.469469
D12      89.041047
D6       87.606767
D8       87.312290
D9       87.312290
V162     86.123717
V142     86.123717
V146     86.123717
V147     86.123717
V141     86.123717
V138     86.123717
V163     86.123717
V161     86.123717
V154     86.123717
V153     86.123717
V158     86.123717
V157     86.123717
V139     86.123717
V148     86.123717
V149     86.123717
V140     86.123717
V156     86.123717
V155     86.123717
V151     86.122701
V159     86.122701
V165     86.122701
V164     86.122701
dtype: float64

In [7]:
print(
    "Transaction IDs in transaction table:",
    transactions["TransactionID"].nunique()
)

print(
    "Transaction IDs in identity table:",
    identity["TransactionID"].nunique()
)

print(
    "Identity IDs also present in transactions:",
    identity["TransactionID"].isin(
        transactions["TransactionID"]
    ).mean() * 100
)

Transaction IDs in transaction table: 590540
Transaction IDs in identity table: 144233
Identity IDs also present in transactions: 100.0


In [8]:
merged = transactions.merge(
    identity,
    on="TransactionID",
    how="left"
)

print("Merged shape:", merged.shape)

Merged shape: (590540, 434)


In [9]:
transactions.groupby("isFraud")["TransactionAmt"].describe()

transactions.groupby("isFraud")["TransactionAmt"].mean()

transactions.groupby("isFraud")["TransactionAmt"].median()

isFraud
0    68.5
1    75.0
Name: TransactionAmt, dtype: float64

In [10]:
transactions.groupby("isFraud")["TransactionDT"].describe()


transactions.sort_values("TransactionDT")[
    ["TransactionID", "TransactionDT", "isFraud"]
].head()



transactions.sort_values("TransactionDT")[
    ["TransactionID", "TransactionDT", "isFraud"]
].tail()

,TransactionID,TransactionDT,isFraud
590535,3577535,15811047,0
590536,3577536,15811049,0
590537,3577537,15811079,0
590538,3577538,15811088,0
590539,3577539,15811131,0


In [11]:
print("TransactionDT min:", transactions["TransactionDT"].min())
print("TransactionDT max:", transactions["TransactionDT"].max())


print(
    transactions.groupby("isFraud")["TransactionDT"]
    .agg(["min", "max", "mean", "median"])
)


print(
    transactions.groupby("isFraud")["TransactionDT"]
    .agg(["min", "max", "mean", "median"])
)

TransactionDT min: 86400
TransactionDT max: 15811131
           min       max          mean     median
isFraud                                          
0        86400  15811131  7.360791e+06  7271678.0
1        89760  15810876  7.690033e+06  7575230.0
           min       max          mean     median
isFraud                                          
0        86400  15811131  7.360791e+06  7271678.0
1        89760  15810876  7.690033e+06  7575230.0


In [12]:
merged_check = transactions[["TransactionID", "isFraud"]].merge(
    identity[["TransactionID"]],
    on="TransactionID",
    how="left",
    indicator=True
)

merged_check["has_identity"] = (
    merged_check["_merge"] == "both"
).astype(int)

print(
    merged_check.groupby("isFraud")["has_identity"]
    .mean()
)

isFraud
0    0.233235
1    0.547742
Name: has_identity, dtype: float64


In [ ]:
import sys
from pathlib import Path

# Add the project root to the path
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

from src.features.feature_engineering import add_basic_features

features_df = add_basic_features(
    transactions,
    identity
)

print(features_df.shape)

ModuleNotFoundError: No module named 'src'